
# Milvus on Zilliz Cloud

This notebook shows **how to connect directly to a cloud‑hosted Milvus database (Zilliz Cloud)** and perform:

- Connection & authentication
- Collection inspection
- CRUD operations
- Vector similarity search
- Visual exploration of embeddings (2D)

Designed for **teaching Milvus concepts**, not local Docker setups.



## 1 Install & Import Dependencies


In [ ]:
%pip install pymilvus


## 2 Connect to Zilliz Cloud (Milvus)



In [ ]:
import os
from dotenv import load_dotenv
# from pymilvus import connections

# # If using Docker standalone Milvus
# connections.connect("default", host="127.0.0.1", port="19530")

from pymilvus import connections

load_dotenv(override=True, dotenv_path="../.env.local")

milvus_uri = os.getenv("MILVUS_URI")
milvus_token = os.getenv("MILVUS_API_KEY")


connections.connect(
    alias="default",
    uri=milvus_uri,
    token=milvus_token
)

print("Connected to Milvus on Zilliz Cloud")



## 3 Inspect Collections


In [ ]:

from pymilvus import utility

utility.list_collections()



## 4 Load & Inspect a Collection


In [ ]:

from pymilvus import Collection

# Collection is same as a Table in traditional databases
collection = Collection("demo_collection")
collection.load()

collection.schema



## 5 Read Data (Query)

Query a few rows to see **raw stored data**


In [ ]:
results = collection.query(
    expr="id >= 0",  
    # output_fields=["id", "title", "vector"],
    output_fields=["id", "title"],
    limit=5
)
# select id, title, vector from demo_collection where id >= 0 limit 5
# select id, title from demo_collection where id >= 0 limit 5
# select <output_fields> from <Collection> where <expression> limit <number>
results


In [ ]:
results = collection.query(
    expr="id in [463705163763347400, 463705164234735154] AND title == 'Deep Learning (Updated)'",
    
    output_fields=["id", "title", "vector"],
    limit=5
)
# select id, title, vector from demo_collection 
# where id in [463705163763347400, 463705164234735154] AND title == 'Deep Learning (Updated)' limit 5
results



## 6 Insert (CREATE)

Insert a new vector record


In [ ]:
import numpy as np

data = [
    [[.3456, .2345, .1234, .5678]],            # vector FIRST
    ["Milvus makes vector search scalable - May 27th 2026"]     # title SECOND
]

collection.insert(data)
collection.flush()



## 7 Update (DELETE + INSERT pattern)

Milvus does not support in‑place updates.


In [ ]:
# collection.delete(expr="id == 463705163763347399")
# collection.flush()

updated_data = [
    [463705165204270561],  # ← list of IDs (1 row)
    [[0.7000895, 0.022113776, 0.48144588, 0.23203984]],  # ← list of vectors (1 row)
    ["Next.JS for Beginners - Updated May 27th 2026"]   # ← list of titles (1 row)
]

result = collection.upsert(updated_data)
collection.flush()

new_id = result.primary_keys[0]
print(f"Record updated. New ID generated: {new_id}")



## 8 Delete


In [ ]:

collection.delete(expr="id == 466305617795297852")
collection.flush()

print("Record deleted")



In [ ]:

results = collection.query(
    expr="id == 463705164234754808",
    output_fields=[ "title", "vector"],
    # output_fields=["*"],
    limit=5
)

results



## 9 Vector Similarity Search


In [ ]:
query_vector = [0.4670895, 0.343513776, 0.22224588, 0.113984]
print(f"Query Vector: {query_vector}")
search_results = collection.search(
    data=[query_vector],
    anns_field="vector",
    param={"metric_type": "COSINE", "params": {"nprobe": 10}},
    limit=2,
    output_fields=["id","title"]
)
# SELECT id, title FROM demo_collection WHERE COSINE_SIMILARITY(vector, [0.4670895, 0.343513776, 0.22224588, 0.113984]) > threshold LIMIT 2
print(f"Search Results: {search_results}")

for hit in search_results[0]:
    print(f"title={hit.entity.get('title')}")
